# Compare Fuzzy vs Cluster-Based Matching

Compare the two R-to-C case matching methods:
- **Fuzzy** (`rc_ac_matches.parquet`) — hybrid mode (exact + fuzzy company name matching)
- **Cluster** (`rc_ac_cluster_matches_20260517.parquet`) — exact matching on cluster representative names, using the cluster assignments file `cluster_assignments_20260517.csv` (singletons absent from this file are treated as their own cluster, represented by their preprocessed name)

In [25]:
import pandas as pd

fuzzy = pd.read_parquet("rc_ac_matches.parquet")
cluster = pd.read_parquet("rc_ac_cluster_matches_20260517.parquet")

print(f"Fuzzy match pairs:   {len(fuzzy):,}")
print(f"Cluster match pairs: {len(cluster):,}")

Fuzzy match pairs:   50,472
Cluster match pairs: 60,731


## Overlap: how many (r_case_number, c_case_number) pairs appear in both?

In [26]:
fuzzy_pairs = set(zip(fuzzy["r_case_number"], fuzzy["c_case_number"]))
cluster_pairs = set(zip(cluster["r_case_number"], cluster["c_case_number"]))

both = fuzzy_pairs & cluster_pairs
fuzzy_only = fuzzy_pairs - cluster_pairs
cluster_only = cluster_pairs - fuzzy_pairs

print(f"In both methods:  {len(both):,}")
print(f"Fuzzy only:       {len(fuzzy_only):,}")
print(f"Cluster only:     {len(cluster_only):,}")
print(f"Union (either):   {len(fuzzy_pairs | cluster_pairs):,}")
print()
print(f"Overlap as % of fuzzy:   {100 * len(both) / len(fuzzy_pairs):.1f}%")
print(f"Overlap as % of cluster: {100 * len(both) / len(cluster_pairs):.1f}%")

In both methods:  49,426
Fuzzy only:       1,046
Cluster only:     11,305
Union (either):   61,777

Overlap as % of fuzzy:   97.9%
Overlap as % of cluster: 81.4%


## RC-case level comparison

How many RC cases got matched by each method?

In [27]:
fuzzy_rc = set(fuzzy["r_case_number"])
cluster_rc = set(cluster["r_case_number"])

print(f"RC cases matched by fuzzy:        {len(fuzzy_rc):,}")
print(f"RC cases matched by cluster:      {len(cluster_rc):,}")
print(f"RC cases matched by both:         {len(fuzzy_rc & cluster_rc):,}")
print(f"RC cases matched by fuzzy only:   {len(fuzzy_rc - cluster_rc):,}")
print(f"RC cases matched by cluster only: {len(cluster_rc - fuzzy_rc):,}")

RC cases matched by fuzzy:        25,867
RC cases matched by cluster:      29,554
RC cases matched by both:         25,441
RC cases matched by fuzzy only:   426
RC cases matched by cluster only: 4,113


## CA-case level comparison

How many CA cases got matched by each method?

In [28]:
fuzzy_ca = set(fuzzy["c_case_number"])
cluster_ca = set(cluster["c_case_number"])

print(f"CA cases matched by fuzzy:        {len(fuzzy_ca):,}")
print(f"CA cases matched by cluster:      {len(cluster_ca):,}")
print(f"CA cases matched by both:         {len(fuzzy_ca & cluster_ca):,}")
print(f"CA cases matched by fuzzy only:   {len(fuzzy_ca - cluster_ca):,}")
print(f"CA cases matched by cluster only: {len(cluster_ca - fuzzy_ca):,}")

CA cases matched by fuzzy:        46,573
CA cases matched by cluster:      55,611
CA cases matched by both:         45,669
CA cases matched by fuzzy only:   904
CA cases matched by cluster only: 9,942


## Inspect fuzzy-only pairs

These pairs were found by fuzzy matching but NOT by clustering. This means the company names were similar enough for fuzzy matching (score >= 82) but were not placed in the same cluster by the LLM.

In [29]:
fuzzy_only_df = fuzzy[
    fuzzy.apply(lambda r: (r["r_case_number"], r["c_case_number"]) in fuzzy_only, axis=1)
].copy()

print(f"Fuzzy-only pairs: {len(fuzzy_only_df):,}")
print(f"\nBy match_method:")
print(fuzzy_only_df["match_method"].value_counts().to_string())
print(f"\nFuzzy score distribution (fuzzy-only pairs):")
print(fuzzy_only_df["fuzzy_score"].describe().round(1).to_string())

Fuzzy-only pairs: 1,046

By match_method:
match_method
fuzzy    1046

Fuzzy score distribution (fuzzy-only pairs):
count    1046.0
mean       88.1
std         4.6
min        82.1
25%        84.2
50%        87.0
75%        91.7
max        98.4


In [30]:
# Sample fuzzy-only pairs: show original and preprocessed names
print("Sample fuzzy-only pairs:\n")
sample = fuzzy_only_df.sample(min(20, len(fuzzy_only_df)), random_state=42)
for _, row in sample.iterrows():
    print(f"  R original:     {row['r_company_name']}")
    print(f"  C original:     {row['c_company_name']}")
    print(f"  R preprocessed: {row['match_company_r']}")
    print(f"  C preprocessed: {row['match_company_c']}")
    print(f"  method={row['match_method']}, score={row['fuzzy_score']:.1f}")
    print()

Sample fuzzy-only pairs:

  R original:     Mirage Hotel and Casino       
  C original:     Riviera Hotel and Casino
  R preprocessed: mirage hotel and casino
  C preprocessed: riviera hotel and casino
  method=fuzzy, score=85.1

  R original:     A&A MECHANICAL, INC.
  C original:     A & A MECHANICAL, INC.
  R preprocessed: aanda mechanical
  C preprocessed: a and a mechanical
  method=fuzzy, score=88.2

  R original:     AIPORT AVIATION SERVICES
  C original:     AIRPORT AVIATION SERVICES, INC.
  R preprocessed: aiport aviation services
  C preprocessed: airport aviation services
  method=fuzzy, score=98.0

  R original:     WILLIAMS ELECTRIC CO. [3]
  C original:     WILLIAMS ELECTRIC COMPANY [IOM] [2]
  R preprocessed: williams electric 3
  C preprocessed: williams electric iom 2
  method=fuzzy, score=85.7

  R original:     AGGREGATE EQUIPMENT & SUPPLY
  C original:     BET SERVICES/AGGREGATE EQUIPMENT & SUPPLY
  R preprocessed: aggregate equipment and supply
  C preprocessed: b

## Inspect cluster-only pairs

These pairs were found by clustering but NOT by fuzzy matching. This means the LLM grouped the company names into the same cluster, but the names were either too different for the fuzzy threshold or differed in city.

In [31]:
cluster_only_df = cluster[
    cluster.apply(lambda r: (r["r_case_number"], r["c_case_number"]) in cluster_only, axis=1)
].copy()

print(f"Cluster-only pairs: {len(cluster_only_df):,}")

Cluster-only pairs: 11,305


In [32]:
# Sample cluster-only pairs: show original and preprocessed (representative) names
print("Sample cluster-only pairs:\n")
sample = cluster_only_df.sample(min(20, len(cluster_only_df)), random_state=42)
for _, row in sample.iterrows():
    print(f"  R original:       {row['r_company_name']}")
    print(f"  C original:       {row['c_company_name']}")
    print(f"  R preprocessed:   {row['match_company_r']}")
    print(f"  C preprocessed:   {row['match_company_c']}")
    print(f"  (cluster representative: {row['match_company_r']})")
    print()

Sample cluster-only pairs:

  R original:       Regal Ready Mix               
  C original:       Regal Materials, Inc., d/b/a Regal Ready Mix
  R preprocessed:   regal ready mix
  C preprocessed:   regal ready mix
  (cluster representative: regal ready mix)

  R original:       RENO HILTON HOTEL & CASINO
  C original:       HILTON HOTELS CORPORATION/DBA/THE RENO HILTON
  R preprocessed:   reno hiltonhere
  C preprocessed:   reno hiltonhere
  (cluster representative: reno hiltonhere)

  R original:       BERLINER & MARX
  C original:       BERLINER & MARX-EDGAR PACKING DIVN.
  R preprocessed:   berliner and marx
  C preprocessed:   berliner and marx
  (cluster representative: berliner and marx)

  R original:       TITAN - S - MFG.  INC.
  C original:       TITAN-S-MFG., INC.
  R preprocessed:   titan s mfg
  C preprocessed:   titan s mfg
  (cluster representative: titan s mfg)

  R original:       Publix Supermarkets, Inc.     
  C original:       Publix Super Markets, Inc.
  R prepr

## Why do "exact" pairs from fuzzy end up as fuzzy-only?

Some pairs found by the exact pass in fuzzy matching might not appear in cluster results. This can happen when two cases have the same preprocessed company name but different cluster representatives. Check how many fuzzy-only pairs were exact matches.

In [33]:
exact_but_fuzzy_only = fuzzy_only_df[fuzzy_only_df["match_method"] == "exact"]
print(f"Fuzzy-only pairs that were 'exact' matches: {len(exact_but_fuzzy_only):,}")

if len(exact_but_fuzzy_only) > 0:
    print("\nSample — same preprocessed name, but cluster didn't unify them:")
    for _, row in exact_but_fuzzy_only.head(10).iterrows():
        print(f"  R: {row['r_company_name']!r}  (preprocessed: {row['match_company_r']!r})")
        print(f"  C: {row['c_company_name']!r}  (preprocessed: {row['match_company_c']!r})")
        print(f"     city R: {row['match_city_r']!r}, city C: {row['match_city_c']!r}")
        print()

Fuzzy-only pairs that were 'exact' matches: 0


## Summary comparison table

In [34]:
summary = pd.DataFrame({
    "Metric": [
        "Total match pairs",
        "Unique RC cases matched",
        "Unique CA cases matched",
        "Pairs in both methods",
        "Pairs only in this method",
        "CA per matched RC (mean)",
        "CA per matched RC (median)",
        "CA per matched RC (max)",
    ],
    "Fuzzy (hybrid)": [
        f"{len(fuzzy):,}",
        f"{len(fuzzy_rc):,}",
        f"{len(fuzzy_ca):,}",
        f"{len(both):,}",
        f"{len(fuzzy_only):,}",
        f"{fuzzy.groupby('r_case_number')['c_case_number'].count().mean():.2f}",
        f"{fuzzy.groupby('r_case_number')['c_case_number'].count().median():.1f}",
        f"{fuzzy.groupby('r_case_number')['c_case_number'].count().max()}",
    ],
    "Cluster (exact)": [
        f"{len(cluster):,}",
        f"{len(cluster_rc):,}",
        f"{len(cluster_ca):,}",
        f"{len(both):,}",
        f"{len(cluster_only):,}",
        f"{cluster.groupby('r_case_number')['c_case_number'].count().mean():.2f}",
        f"{cluster.groupby('r_case_number')['c_case_number'].count().median():.1f}",
        f"{cluster.groupby('r_case_number')['c_case_number'].count().max()}",
    ],
})
summary

,Metric,Fuzzy (hybrid),Cluster (exact)
0,Total match pairs,"50,472","60,731"
1,Unique RC cases matched,"25,867","29,554"
2,Unique CA cases matched,"46,573","55,611"
3,Pairs in both methods,"49,426","49,426"
4,Pairs only in this method,"1,046","11,305"
5,CA per matched RC (mean),1.95,2.05
6,CA per matched RC (median),1.0,1.0
7,CA per matched RC (max),272,272


## Non-trivial subset (preprocessed names differ)

Both methods operate on **preprocessed** company names. If the preprocessed
names of an RC and a CA case are identical, both methods trivially link them —
this counts as "agreement" but doesn't measure any real entity-resolution work.

About 80% of agreement pairs are trivial (identical preprocessed names). The
overlap percentages above include these trivial pairs and so overstate how much
the two methods actually agree on **interesting** cases.

The cells below recompute the overlap restricted to **non-trivial** pairs:
those where preprocessed names differ. This is the more meaningful comparison
of the two methods' entity-resolution capabilities.

(`fuzzy_only` and `cluster_only` are structurally guaranteed non-trivial —
if preprocessed names were identical, both methods would match the pair.
Only the agreement cell needs filtering.)


In [35]:
from preprocessing_v3 import preprocess_employer
from name_standardization import standardize_company_name

def _preprocess(n):
    if pd.isna(n):
        return ""
    return standardize_company_name(preprocess_employer(str(n)))

# Fuzzy: match_company_r / match_company_c ARE the preprocessed names already
fuzzy_nontrivial = fuzzy[fuzzy["match_company_r"] != fuzzy["match_company_c"]].copy()

# Cluster: match_company_r is the cluster REPRESENTATIVE, not the preprocessed name.
# We need to re-preprocess raw company names to filter trivial pairs.
cluster = cluster.copy()
cluster["_pp_r"] = cluster["r_company_name"].apply(_preprocess)
cluster["_pp_c"] = cluster["c_company_name"].apply(_preprocess)
cluster_nontrivial = cluster[cluster["_pp_r"] != cluster["_pp_c"]].copy()

print(f"Fuzzy pairs:    {len(fuzzy):,}  ->  non-trivial: {len(fuzzy_nontrivial):,}  "
      f"({100 * len(fuzzy_nontrivial) / len(fuzzy):.1f}%)")
print(f"Cluster pairs:  {len(cluster):,}  ->  non-trivial: {len(cluster_nontrivial):,}  "
      f"({100 * len(cluster_nontrivial) / len(cluster):.1f}%)")


Fuzzy pairs:    50,472  ->  non-trivial: 10,734  (21.3%)
Cluster pairs:  60,731  ->  non-trivial: 20,993  (34.6%)


In [36]:
fuzzy_nt_pairs   = set(zip(fuzzy_nontrivial.r_case_number, fuzzy_nontrivial.c_case_number))
cluster_nt_pairs = set(zip(cluster_nontrivial.r_case_number, cluster_nontrivial.c_case_number))

both_nt         = fuzzy_nt_pairs & cluster_nt_pairs
fuzzy_only_nt   = fuzzy_nt_pairs - cluster_nt_pairs
cluster_only_nt = cluster_nt_pairs - fuzzy_nt_pairs

print("Non-trivial overlap:")
print(f"  In both:        {len(both_nt):,}")
print(f"  Fuzzy only:     {len(fuzzy_only_nt):,}")
print(f"  Cluster only:   {len(cluster_only_nt):,}")
print(f"  Union:          {len(fuzzy_nt_pairs | cluster_nt_pairs):,}")
print()
print(f"  Overlap as % of fuzzy (non-trivial):   {100 * len(both_nt) / len(fuzzy_nt_pairs):.1f}%")
print(f"  Overlap as % of cluster (non-trivial): {100 * len(both_nt) / len(cluster_nt_pairs):.1f}%")


Non-trivial overlap:
  In both:        9,688
  Fuzzy only:     1,046
  Cluster only:   11,305
  Union:          22,039

  Overlap as % of fuzzy (non-trivial):   90.3%
  Overlap as % of cluster (non-trivial): 46.1%


### All-pairs vs. non-trivial side-by-side

In [37]:
def pct(a, b):
    return f"{100 * a / b:.1f}%" if b else "n/a"

n_fuzzy = len(fuzzy)
n_cluster = len(cluster)
n_both_all = len(set(zip(fuzzy.r_case_number, fuzzy.c_case_number))
                 & set(zip(cluster.r_case_number, cluster.c_case_number)))

n_fuzzy_nt = len(fuzzy_nontrivial)
n_cluster_nt = len(cluster_nontrivial)
n_both_nt = len(both_nt)

comparison = pd.DataFrame({
    "Metric": [
        "Fuzzy match pairs",
        "Cluster match pairs",
        "Pairs in both methods",
        "Overlap as % of fuzzy",
        "Overlap as % of cluster",
    ],
    "All pairs": [
        f"{n_fuzzy:,}",
        f"{n_cluster:,}",
        f"{n_both_all:,}",
        pct(n_both_all, n_fuzzy),
        pct(n_both_all, n_cluster),
    ],
    "Non-trivial only": [
        f"{n_fuzzy_nt:,}",
        f"{n_cluster_nt:,}",
        f"{n_both_nt:,}",
        pct(n_both_nt, n_fuzzy_nt),
        pct(n_both_nt, n_cluster_nt),
    ],
})
comparison


,Metric,All pairs,Non-trivial only
0,Fuzzy match pairs,"50,472","10,734"
1,Cluster match pairs,"60,731","20,993"
2,Pairs in both methods,"49,426","9,688"
3,Overlap as % of fuzzy,97.9%,90.3%
4,Overlap as % of cluster,81.4%,46.1%
